# 検索した文書を LLM に渡して答えさせる (最小の RAG)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_llm/rag_minimal.ipynb)

NLP クックブック 12-09 ／ 大規模言語モデル (LLM)

LLM が知らない情報 (社内文書、最新情報) について、関連文書を検索してから答えさせたい。

In [1]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

docs = [
    "クックブック食堂の営業時間は平日 11 時から 15 時、土日は 11 時から 20 時です。月曜日は定休日です。",
    "クックブック食堂の看板メニューは、3 日間煮込んだ特製ビーフカレー (950 円) です。大盛りは 100 円増しです。",
    "クックブック食堂は 2019 年に京都市左京区で開業しました。店主の田中さんはインドで 2 年間修行しました。",
    "駐車場は店舗の裏に 3 台分あります。満車の場合は近隣のコインパーキングをご利用ください。",
    "予約は電話 (075-000-0000) のみ受け付けています。5 名以上の場合は前日までにご連絡ください。",
    "支払いは現金と QR コード決済に対応しています。クレジットカードは使えません。",
]
embedder = SentenceTransformer("intfloat/multilingual-e5-small")
doc_emb = embedder.encode(["passage: " + d for d in docs], normalize_embeddings=True)

name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def generate(prompt):
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=80, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [2]:
def retrieve(question, k=2):
    sims = doc_emb @ embedder.encode("query: " + question, normalize_embeddings=True)
    return [(int(i), docs[i]) for i in sims.argsort()[::-1][:k]]

def rag(question):
    hits = retrieve(question)
    context = "\n".join(f"[{i}] {d}" for i, d in hits)
    prompt = (f"以下の資料だけに基づいて質問に答えてください。資料に答えがなければ「わかりません」と答えてください。\n\n"
              f"# 資料\n{context}\n\n# 質問\n{question}")
    return generate(prompt), [i for i, _ in hits]

for q in ["カレーはいくらですか？", "日曜日は何時まで開いていますか？", "クレジットカードで払えますか？"]:
    answer, sources = rag(q)
    print(f"Q: {q}\n  検索なし: {generate(q)[:60]}\n  RAG: {answer} (参照: {sources})")

Q: カレーはいくらですか？
  検索なし: カレーの値段は、その具材やブランド、地域、お店によって大きく異なります。一般的に、中辛から甘口までのカレールーは、1杯あ
  RAG: カレーの金額は、資料に記載されている通り950円です。これは特製ビーフカレーで、大盛りにすると100円増しになります。

（回答：カレーの金額は確かに950円です。大盛りにするとさらに100円増しとなります。） (参照: [1, 5])
Q: 日曜日は何時まで開いていますか？
  検索なし: 日曜日は通常24時（深夜0時）まで開いているお店や施設はほとんどありません。しかし、一部の店舗やサービス機関によっては、
  RAG: 日曜日は営業しており、通常であれば20時まで開いていることが多いです。しかし、具体的な営業時間はクックブック食堂の実際のスケジュールによって異なる場合があるので、正確な情報を得るためには直接店舗に問い合わせるのが最も確実です。もし資料に答えがなければ、「わかりません」と答えることになります。 (参照: [0, 3])
Q: クレジットカードで払えますか？
  検索なし: はい、クレジットカードでの支払いは一般的に可能です。多くのクレジットカードブランド（例えば、American Expre
  RAG: いいえ、このレストランではクレジットカードは使用できません。支払いは現金とQRコード決済に対応していますが、クレジットカードの利用はできません。 (参照: [5, 4])


- RAG = 検索 (Retrieval) + 生成 (Generation)。検索なしだと、モデルは知らない店についてもっともらしい嘘 (ハルシネーション) を答える。
- 精度の大半は検索で決まる。まず「正解の文書が上位 k 件に入っているか」を評価し、ハイブリッド検索やリランキングで改善する。
- 参照した文書の番号を一緒に返すと、利用者が根拠を確認できる。